# Image filtering with VisionQL v0.1

Query `data/datasets/images/coco128/images` through a vectorized Python UDF and an ONNX model, then display the Arrow result.

In [ ]:
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import visionql

In [ ]:
def quality_score(images):
    # IMAGE crosses the Python boundary in encoded form and stays vectorized.
    return pc.cast(pc.is_valid(images.field("encoded")), pa.float32())

In [ ]:
repo = Path.cwd()
if not (repo / "vql-kernel").exists():
    repo = repo.parent.parent
images = (repo / "data/datasets/images/coco128/images").resolve()
model = (repo / "data/models/yolo26n.onnx").resolve()
if not model.is_file():
    raise FileNotFoundError("Run 'python scripts/export_yolo26.py --size n' first")
session = visionql.connect()

In [ ]:
session.sql(f"CREATE TABLE product_photos USING IMAGES LOCATION '{images}' WITH (recursive=true)").collect()
session.sql(
    "CREATE FUNCTION quality(img IMAGE) RETURNS FLOAT "
    "LANGUAGE PYTHON AS '__main__:quality_score'"
).collect()
session.sql(
    f"CREATE MODEL yolo26n TYPE OBJECT_DETECTION FROM 'file://{model}' "
    "WITH (processor='yolo26-detect-v1')"
).collect()
session.sql(
    "CREATE FUNCTION detect USING MODEL yolo26n "
    "WITH (classes=['person'], min_confidence=0.6)"
).collect()

In [ ]:
result = session.sql(
    "SELECT uri, COUNT_OBJECTS(detect(image), 'person', 0.6) AS people "
    "FROM product_photos WHERE quality(image) >= 0 ORDER BY uri"
)
result.collect()